# Pyannote Diarization testing - pyannote/speaker-diarization-community-1
# <https://huggingface.co/pyannote/speaker-diarization-community-1>

In [2]:
# setup python system path if needed
import sys, json
from pathlib import Path

parent_dir = Path.cwd().resolve().parent  # parent = project root
ffmpeg_bin_path = parent_dir / "ffmpeg" / "bin"

sys.path.append(str(parent_dir))
sys.path.append(str(ffmpeg_bin_path))


def print_formatted_sys_path():
    # Rather than deal with the raw output of sys.path, we can use the json module to spit out a nicely formatted object
    formatted_path = json.dumps(sys.path, indent=4)

    # use some colors to make it pretty!
    print("\033[1;34m[SYS.PATH]\033[0m")  # ANSI code for blue for the title
    print("\033[1;32m" + formatted_path + "\033[0m")  # ANSI code for green for paths, then ANSI code for reset


# Call the function to print the formatted sys.path
print_formatted_sys_path()


[SYS.PATH]
[
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.14.2-windows-x86_64-none\\python314.zip",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.14.2-windows-x86_64-none\\DLLs",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.14.2-windows-x86_64-none\\Lib",
    "D:\\Users\\andrew.sparkes\\AppData\\Roaming\\uv\\python\\cpython-3.14.2-windows-x86_64-none",
    "z:\\code\\STAT405_AudioNotetaker\\.venv",
    "",
    "z:\\code\\STAT405_AudioNotetaker\\.venv\\Lib\\site-packages",
    "\\\\zdrive.labs.cset.oit.edu\\zdrive\\andrew.sparkes\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin",
    "\\\\zdrive.labs.cset.oit.edu\\zdrive\\andrew.sparkes\\code\\STAT405_AudioNotetaker",
    "\\\\zdrive.labs.cset.oit.edu\\zdrive\\andrew.sparkes\\code\\STAT405_AudioNotetaker\\ffmpeg\\bin"
]


In [3]:
# user configurable settings
audio_file = Path("sample_data/en_US/Gene_Hackman_Interview_by_Bob_Lardine.mp4")


In [7]:
# setup stuff
import log_config # to override default and use loguru instead
log_config.setup_logging()
from loguru import logger

from pydantic_settings import BaseSettings, SettingsConfigDict

class NotebookSettings(BaseSettings):
    hf_token: str
    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8", extra="allow")

settings = NotebookSettings()

#print(f"HF_TOKEN env variable: {settings.hf_token}")


Looks like the `HF_TOKEN` environment variable is reading properly, huzzah, let's move on...

This pipeline ingests mono audio sampled at 16kHz and outputs speaker diarization.

 - stereo or multi-channel audio files are automatically downmixed to mono by averaging the channels.
 - audio files sampled at a different rate are resampled to 16kHz automatically upon loading.

The main improvements brought by Community-1 are:

 - improved speaker assignment and counting
 - simpler reconciliation with transcription timestamps with exclusive speaker diarization
 - easy offline use (i.e. without internet connection)
 - (optionally) hosted on pyannoteAI cloud


### Exclusive speaker diarization

Community-1 pre-trained pipeline returns a new exclusive speaker diarization, on top of the regular speaker diarization, available as `output.exclusive_speaker_diarization`.

This is a feature which is backported from our latest commercial model that simplifies the reconciliation between fine-grained speaker diarization timestamps and (sometimes not so precise) transcription timestamps.

In [ ]:
import hf


In [ ]:
%%bash
uvx hf auth login


In [8]:
# pytorch setup device
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")


Using device: cpu


In [9]:
# example usage from HF page use model dialog

from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1")

# sets the pipeline to use cuda over cpu
pipeline.to(device)

# inference on the whole file
pipeline(str(audio_file))
#pipeline(audio_file.resolve()) # maybe?

# # inference on an excerpt
# from pyannote.core import Segment
# excerpt = Segment(start = 2.0, end = 5.0)

# from pyannote.audio import Audio
# waveform, sample_rate = Audio().crop("file.wav", excerpt)
# pipeline({"waveform": waveform, "sample_rate": sample_rate})


z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             tab

config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

: 

: 

In [ ]:
# example usage from HF page itself

# download the pipeline from Huggingface
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1", 
    token = settings.hf_token
    )


# sets the pipeline to use cuda over cpu
pipeline.to(device)


# run the pipeline locally on your computer
output = pipeline(str(audio_file))
# output = pipeline(audio_file.resolve()) # maybe?

# print the predicted speaker diarization 
for turn, speaker in output.speaker_diarization:
    print(f"{speaker} speaks between t={turn.start:.3f}s and t={turn.end:.3f}s")
    
print(f"-"*80)

output


## Offline use

In the terminal, copy the pipeline on disk:


In [ ]:
# make sure git-lfs is installed (https://git-lfs.com)
git lfs install

# create a directory on disk
mkdir pipeline

# when prompted for a password, use an access token with write permissions.
# generate one from your settings: https://huggingface.co/settings/tokens
git clone https://hf.co/pyannote/speaker-diarization-community-1 pipeline/pyannote-speaker-diarization-community-1


In Python, use the pipeline without internet connection:


In [ ]:
# load pipeline from disk (works without internet connection)
from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained("pipeline/pyannote-speaker-diarization-community-1")

# run the pipeline locally on your computer
output = pipeline(str(audio_file))
